In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Full RFM Segmentation and Customer Profiling.csv', encoding='latin-1')
print(df.head(5))

   ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER    SALES  \
0        10107               30      95.70                2  2871.00   
1        10121               34      81.35                5  2765.90   
2        10134               41      94.74                2  3884.34   
3        10145               45      83.26                6  3746.70   
4        10159               49     100.00               14  5205.27   

         ORDERDATE   STATUS  QTR_ID  MONTH_ID  YEAR_ID  ...  \
0   2/24/2003 0:00  Shipped       1         2     2003  ...   
1    5/7/2003 0:00  Shipped       2         5     2003  ...   
2    7/1/2003 0:00  Shipped       3         7     2003  ...   
3   8/25/2003 0:00  Shipped       3         8     2003  ...   
4  10/10/2003 0:00  Shipped       4        10     2003  ...   

                    ADDRESSLINE1  ADDRESSLINE2           CITY STATE  \
0        897 Long Airport Avenue           NaN            NYC    NY   
1             59 rue de l'Abbaye           NaN

In [48]:
print('Number of the duplicated values: ', df.duplicated().sum().sum())
print('\nNumber of Null values: ', df.isnull().sum().sum())
print('\nNull values: ', df.isnull().sum())
print('\nDataTpes: ', df.dtypes)

Number of the duplicated values:  0

Number of Null values:  5157

Null values:  ORDERNUMBER            0
QUANTITYORDERED        0
PRICEEACH              0
ORDERLINENUMBER        0
SALES                  0
ORDERDATE              0
STATUS                 0
QTR_ID                 0
MONTH_ID               0
YEAR_ID                0
PRODUCTLINE            0
MSRP                   0
PRODUCTCODE            0
CUSTOMERNAME           0
PHONE                  0
ADDRESSLINE1           0
ADDRESSLINE2        2521
CITY                   0
STATE               1486
POSTALCODE            76
COUNTRY                0
TERRITORY           1074
CONTACTLASTNAME        0
CONTACTFIRSTNAME       0
DEALSIZE               0
dtype: int64

DataTpes:  ORDERNUMBER           int64
QUANTITYORDERED       int64
PRICEEACH           float64
ORDERLINENUMBER       int64
SALES               float64
ORDERDATE            object
STATUS               object
QTR_ID                int64
MONTH_ID              int64
YEAR_ID          

In [49]:
#convert ORDERDATE to datetime
df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'], format='%m/%d/%Y %H:%M')
print('\nDateTypes of OrderDate: ', df['ORDERDATE'].dtypes)
print('\n',df['ORDERDATE'].head(5))


DateTypes of OrderDate:  datetime64[ns]

 0   2003-02-24
1   2003-05-07
2   2003-07-01
3   2003-08-25
4   2003-10-10
Name: ORDERDATE, dtype: datetime64[ns]


In [50]:
from datetime import timedelta
#Set reference_date = df['ORDERDATE'].max() + timedelta(days=1)
reference_date = df['ORDERDATE'].max() + timedelta(days=1)
print(reference_date)

2005-06-01 00:00:00


In [51]:
#Calculate per CUSTOMERNAME:

#- Recency: days since last order
#- Frequency: number of unique orders (ORDERNUMBER)
#- Monetary: total SALES value

df['FullName'] = df['CONTACTFIRSTNAME'] + ' ' + df['CONTACTLASTNAME']

RFM = df.groupby('FullName').agg(
    Recency=('ORDERDATE', lambda x: (reference_date - x.max()).days),
    Frequency=('ORDERNUMBER', 'nunique'),
    Monetary=('SALES', 'sum')
).sort_values(by='Monetary', ascending=False)

print(RFM.head(5))

                Recency  Frequency   Monetary
FullName                                     
Diego Freyre          1         26  912294.11
Valarie Nelson        3         17  654858.06
Peter Ferguson      184          5  200995.41
Jeff Young          182          4  197736.94
Janine Labrune        1          4  180124.90


In [52]:
#Score each metric 1–4 using pd.qcut (4 = best for F and M; 4 = most recent for R)

#ən son nə zaman alış-veriş edib
RFM['R_Score'] = pd.qcut(RFM['Recency'].rank(method='first'), q=4, labels=[4, 3, 2, 1])

#nə qədər tezliklə edir
RFM['F_Score'] = pd.qcut(RFM['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])

#nə qədər xərcləyi
RFM['M_Score'] = pd.qcut(RFM['Monetary'].rank(method='first'), q=4, labels=[1, 2, 3, 4])

print(RFM[['R_Score', 'F_Score', 'M_Score']].head())

               R_Score F_Score M_Score
FullName                              
Diego Freyre         4       4       4
Valarie Nelson       4       4       4
Peter Ferguson       3       4       4
Jeff Young           3       4       4
Janine Labrune       4       4       4


In [53]:
#Create RFM_Segment by concatenating R, F, M scores as strings (e.g. "444", "311")

RFM['RFM_Segment'] = (
    RFM['R_Score'].astype(str) +
    RFM['F_Score'].astype(str) +
    RFM['M_Score'].astype(str)
)

print(RFM[['R_Score', 'F_Score', 'M_Score', 'RFM_Segment']].head(5))

               R_Score F_Score M_Score RFM_Segment
FullName                                          
Diego Freyre         4       4       4         444
Valarie Nelson       4       4       4         444
Peter Ferguson       3       4       4         344
Jeff Young           3       4       4         344
Janine Labrune       4       4       4         444


In [54]:
#Define and assign at least 5 named segments:

RFM['Segment_Code'] = RFM['R_Score'].astype(str) + RFM['F_Score'].astype(str)

map = {
    r'[3-4][3-4]': 'Champions',
    r'[3-4][1-2]': 'Loyal',
    r'2[1-4]': 'At Risk',
    r'1[3-4]': 'Need Attention',
    r'1[1-2]': 'Lost'
}

RFM['RFM_Segment_Name'] = RFM['Segment_Code'].replace(map, regex=True)

print(RFM[['R_Score', 'F_Score', 'M_Score', 'RFM_Segment_Name']].head())

               R_Score F_Score M_Score RFM_Segment_Name
FullName                                               
Diego Freyre         4       4       4        Champions
Valarie Nelson       4       4       4        Champions
Peter Ferguson       3       4       4        Champions
Jeff Young           3       4       4        Champions
Janine Labrune       4       4       4        Champions


In [55]:
#Calculate segment sizes and average Monetary per segment

segment_analysis = RFM.groupby('RFM_Segment_Name').agg({
    'Monetary': ['count', 'mean']
})

segment_analysis.columns = ['Segment_Size', 'Monetary_Average']

segment_analysis = segment_analysis.sort_values(by='Monetary_Average', ascending=False).round(2)

print(segment_analysis)

                  Segment_Size  Monetary_Average
RFM_Segment_Name                                
Champions                   35         145806.17
Loyal                       11         104355.62
At Risk                     23          93393.70
Lost                        23          71019.39


In [64]:
#Build a summary table: segment name, customer count, avg recency, avg frequency, avg monetary

summary_table = RFM.groupby('RFM_Segment_Name').agg(
    Customer_Count=('Recency', 'count'),
    AVG_Recency=('Recency', 'mean'),
    AVG_Frequency=('Frequency', 'mean'),
    AVG_Monetary=('Monetary', 'mean')
)
summary_table = summary_table.sort_values(by='AVG_Monetary', ascending=False).round(2)

print(summary_table)

                  Customer_Count  AVG_Recency  AVG_Frequency  AVG_Monetary
RFM_Segment_Name                                                          
Champions                     35        79.14           4.63     145806.17
Loyal                         11        96.36           2.91     104355.62
At Risk                       23       205.96           2.87      93393.70
Lost                          23       358.83           2.04      71019.39
